In [19]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np


url = "https://raw.githubusercontent.com/plotly/Figure-Friday/refs/heads/main/2025/week-50/AI%20Model%20Hallucination%20Scores.csv"
try:
    df = pd.read_csv(url)
except Exception as e:
    print(f"Error loading URL: {e}. Please check the connection or URL.")
    raise

df.columns = ['Model', 'Accuracy', 'Hallucination', 'Type', 'Owner']
for col in ['Accuracy', 'Hallucination']:
    df[col] = df[col].astype(str).str.replace('%', '', regex=False).str.strip().astype(float)

df_long = pd.melt(
    df, 
    id_vars=['Model', 'Owner', 'Type'], 
    value_vars=['Hallucination', 'Accuracy'], 
    var_name='Metric', 
    value_name='Score'
)

df_long['Metric'] = df_long['Metric'].replace({
    'Accuracy': 'Model Accuracy Index (Higher is Better)',
    'Hallucination': 'Hallucination Index (Lower is Better)'
})
fig_radar = go.Figure()
hallucination_data = df_long[df_long['Metric'] == 'Hallucination Index (Lower is Better)']
fig_radar.add_trace(go.Scatterpolar(
    r=hallucination_data['Score'],
    theta=hallucination_data['Model'],
    fill='toself',
    name='Hallucination Index (Lower is Better)',
    fillcolor='rgba(255, 99, 71, 0.2)', 
    line_color='rgb(255, 99, 71)',
    line_width=2,
    mode='lines+markers',
    marker=dict(size=6, symbol='circle-open', line_width=1.5),
    hovertemplate='<b>Model:</b> %{theta}<br><b>Metric:</b> Hallucination Index<br><b>Score:</b> %{r:.1f} %<extra></extra>'
))
accuracy_data = df_long[df_long['Metric'] == 'Model Accuracy Index (Higher is Better)']
fig_radar.add_trace(go.Scatterpolar(
    r=accuracy_data['Score'],
    theta=accuracy_data['Model'],
    fill='toself',
    name='Model Accuracy Index (Higher is Better)',
    fillcolor='rgba(70, 130, 180, 0.3)', 
    line_color='rgb(70, 130, 180)',
    line_width=3,
    mode='lines+markers',
    marker=dict(size=7, symbol='circle', line_width=0.5),
    hovertemplate='<b>Model:</b> %{theta}<br><b>Metric:</b> Accuracy Index<br><b>Score:</b> %{r:.1f} %<extra></extra>'
))

fig_radar.update_layout(
    polar=dict(
        bgcolor='rgba(0,0,0,0)', 
        radialaxis=dict(
            visible=True, 
            range=[0, 100], 
            gridcolor='lightgray', 
            tickvals=np.arange(0, 101, 10),
            showticklabels=False, 
            title="", 
        ),
        angularaxis=dict(
             tickvals=df['Model'].tolist(), 
             ticktext=df['Model'].tolist(),
             showticklabels=True, 
             ticks="", 
             showline=False,
             period=len(df['Model'].unique())
        )
    ),
    plot_bgcolor='white', 
    title_text='AI Model Performance: Accuracy vs. Hallucination Raw Scores',
    legend_title_text="Model Metric", 
    height=650, 
    width=1000,
    font=dict(size=11),
    margin=dict(t=120,b=30),
    template = "plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=0.8)
)
fig_radar.show()